In [1]:
import pandas as pd

RAW = "../data/raw"
PROCESSED = "../data/processed"

# CPI-U annual averages (BLS), used to convert 2019 dollars to 2024 dollars
CPI_2019 = 255.657
CPI_2024 = 313.689

In [2]:
acs_2019 = pd.read_csv(f"{RAW}/acs_2019.csv", dtype={"state": str, "county": str})
acs_2024 = pd.read_csv(f"{RAW}/acs_2024.csv", dtype={"state": str, "county": str})

acs = pd.concat([acs_2019, acs_2024], ignore_index=True)
print(acs.shape)
acs.head()

(48, 21)


,NAME,median_hh_income,pov_universe,pov_below,renters_total,rent_30_35,rent_35_40,rent_40_50,rent_50_plus,rent_not_computed,...,edu_bachelors,edu_masters,edu_professional,edu_doctorate,households_total,hh_broadband,state,county,pct_uninsured,year
0,"Worcester County, Maryland",63499,50918,4606,5507,421,313,681,1154,382,...,7309,3333,631,167,22089,18276,24,047,4.9,2019
1,"Talbot County, Maryland",73547,36689,3179,4998,171,460,698,956,513,...,5878,3312,1301,847,16826,14490,24,041,4.3,2019
2,"Howard County, Maryland",121160,316280,15805,30650,2287,1681,2063,6703,1336,...,66899,46862,10521,10416,114170,107150,24,027,3.9,2019
3,"Prince George's County, Maryland",84920,888369,75341,118080,11490,8829,10904,27254,4165,...,115214,68683,10115,9844,311343,271353,24,033,10.1,2019
4,"Anne Arundel County, Maryland",100798,554326,32087,54197,5527,3970,4427,10492,2183,...,94465,52737,9937,7164,209814,190839,24,003,4.4,2019


In [3]:
acs["fips"] = acs["state"] + acs["county"]
acs["county_name"] = acs["NAME"].str.replace(", Maryland", "", regex=False)

id_cols = ["NAME", "state", "county", "fips", "county_name", "year"]
count_cols = [c for c in acs.columns if c not in id_cols]

acs[count_cols] = acs[count_cols].apply(pd.to_numeric, errors="coerce")
acs[count_cols] = acs[count_cols].mask(acs[count_cols] < 0)

print(acs[count_cols].isna().sum().sum(), "missing values")

0 missing values


In [4]:
acs["poverty_rate"] = acs["pov_below"] / acs["pov_universe"] * 100

burdened = acs[["rent_30_35", "rent_35_40", "rent_40_50", "rent_50_plus"]].sum(axis=1)
acs["rent_burden_rate"] = burdened / (acs["renters_total"] - acs["rent_not_computed"]) * 100

degrees = acs[["edu_bachelors", "edu_masters", "edu_professional", "edu_doctorate"]].sum(axis=1)
acs["bachelors_plus_rate"] = degrees / acs["pop_25_plus"] * 100

acs["broadband_rate"] = acs["hh_broadband"] / acs["households_total"] * 100

In [5]:
inflation_factor = acs["year"].map({2019: CPI_2024 / CPI_2019, 2024: 1.0})
acs["median_income_2024usd"] = acs["median_hh_income"] * inflation_factor

In [6]:
acs[["county_name", "year", "median_hh_income", "median_income_2024usd"]].head()

,county_name,year,median_hh_income,median_income_2024usd
0,Worcester County,2019,63499,77912.741724
1,Talbot County,2019,73547,90241.553656
2,Howard County,2019,121160,148662.306293
3,Prince George's County,2019,84920,104196.129502
4,Anne Arundel County,2019,100798,123678.302656


In [7]:
preview = pd.read_excel(f"{RAW}/laucnty24.xlsx", header=None)
preview.head(8)

,0,1,2,3,4,5,6,7,8
0,"Labor Force Data by County, 2024 Annual Averages",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,LAUS Code,State FIPS Code,County FIPS Code,County Name/State Abbreviation,Year,Labor Force,Employed,Unemployed,Unemployment Rate (%)
2,CN0100100000000,01,001,"Autauga County, AL",2024,28819,28076,743,2.6
3,CN0100300000000,01,003,"Baldwin County, AL",2024,117566,114409,3157,2.7
4,CN0100500000000,01,005,"Barbour County, AL",2024,8699,8347,352,4
5,CN0100700000000,01,007,"Bibb County, AL",2024,8826,8570,256,2.9
6,CN0100900000000,01,009,"Blount County, AL",2024,26971,26281,690,2.6
7,CN0101100000000,01,011,"Bullock County, AL",2024,4215,4060,155,3.7


In [8]:
def read_bls(path, year):
    raw = pd.read_excel(path, header=None)
    state = pd.to_numeric(raw[1], errors="coerce")
    md = raw[state == 24]
    county = pd.to_numeric(md[2]).astype(int).astype(str).str.zfill(3)
    return pd.DataFrame({
        "fips": "24" + county,
        "year": year,
        "unemployment_rate": pd.to_numeric(md.iloc[:, -1], errors="coerce"),
    })

bls = pd.concat([
    read_bls(f"{RAW}/laucnty19.xlsx", 2019),
    read_bls(f"{RAW}/laucnty24.xlsx", 2024),
], ignore_index=True)

print(bls.shape)
bls.head()

(48, 3)


,fips,year,unemployment_rate
0,24001,2019,5.3
1,24003,2019,3.0
2,24005,2019,3.6
3,24009,2019,3.1
4,24011,2019,3.7


In [9]:
indicators = acs.merge(bls, on=["fips", "year"], how="left")

indicators = indicators.rename(columns={"pct_uninsured": "uninsured_rate"})

final_cols = [
    "fips", "county_name", "year",
    "median_income_2024usd", "poverty_rate", "rent_burden_rate",
    "bachelors_plus_rate", "broadband_rate", "uninsured_rate", "unemployment_rate",
]
indicators = indicators[final_cols].round(2).sort_values(["year", "county_name"])

print(indicators.shape)
print(indicators.isna().sum())
indicators.head()

(48, 10)
fips                     0
county_name              0
year                     0
median_income_2024usd    0
poverty_rate             0
rent_burden_rate         0
bachelors_plus_rate      0
broadband_rate           0
uninsured_rate           0
unemployment_rate        0
dtype: int64


,fips,county_name,year,median_income_2024usd,poverty_rate,rent_burden_rate,bachelors_plus_rate,broadband_rate,uninsured_rate,unemployment_rate
23,24001,Allegany County,2019,56310.33,16.55,51.43,18.89,75.45,4.8,5.3
4,24003,Anne Arundel County,2019,123678.30,5.79,46.94,41.75,90.96,4.4,3.0
5,24005,Baltimore County,2019,94313.94,9.05,49.46,39.36,86.27,5.2,3.6
20,24510,Baltimore city,2019,61814.61,21.21,51.18,31.91,74.48,6.6,4.6
7,24009,Calvert County,2019,134126.14,4.61,49.51,32.41,90.15,3.9,3.1


In [10]:
indicators.describe().round(1)

,year,median_income_2024usd,poverty_rate,rent_burden_rate,bachelors_plus_rate,broadband_rate,uninsured_rate,unemployment_rate
count,48.0,48.0,48.0,48.0,48.0,48.0,48.0,48.0
mean,2021.5,95895.5,10.2,49.9,34.2,87.0,5.2,3.6
std,2.5,27682.0,4.6,5.3,11.8,6.1,1.7,0.8
min,2019.0,46384.0,3.9,35.0,14.4,73.7,2.9,2.5
25%,2019.0,71395.6,6.7,47.7,25.5,83.5,3.9,2.9
50%,2021.5,93160.5,9.2,49.6,35.2,87.9,5.0,3.4
75%,2024.0,119563.0,12.3,52.0,40.4,91.5,6.0,3.9
max,2024.0,149763.0,21.7,67.4,64.5,96.6,11.4,6.8


In [11]:
indicators.to_csv(f"{PROCESSED}/county_indicators.csv", index=False)
print("Saved county_indicators.csv")

Saved county_indicators.csv


# 02 - Clean Data
Combines the raw ACS and BLS files, calculates each indicator as a percentage, adjusts 2019 income for inflation, and saves one tidy table to 'data/processed/'.